# 01 — PDF text

How well the system answers questions about prose documents, the kind of material it was built for. Two measurements:

1. **QASPER** (Dasigi et al., 2021), a published benchmark of questions about research papers with short answers and labelled evidence paragraphs. It gives an answer score and a retrieval score in the benchmark's own terms, set against the published baselines.
2. **Our own 25 questions** over four papers, run through the configuration the application ships, with every wrong answer sorted into a retrieval or a generation failure.

QASPER also has an unanswerable class, so it is where the system's *I don't know* behaviour is judged (the abstention results below).

**Running it.** Every section below is the evaluation's own code. With `RUN = False`
(the default) nothing is recomputed: the results saved in `data/eval/` are loaded
and shown. Set `RUN = True` in the first code cell to measure again, which
overwrites those files. QASPER's full dev set takes about 12 minutes with Qwen2.5-1.5B on the Arc GPU (set `--papers` to 30 for a 2-minute sample); the 25-question analysis a few minutes.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## QASPER

Runs the system on QASPER (Dasigi et al., 2021), the benchmark that matches
what this architecture was built to do.

The benchmark measured before this one was the wrong instrument. It asked for
the colour of a node in a diagram, the sum of a column, whether one bar is
taller than another — questions for a vision-language model reading page
images. This is a text retrieval pipeline,
and a fifth of what it lost there was never in the page text at all.

QASPER asks what a reader of a research paper asks. Its 5,049 questions were
written by people who had seen only a paper's title and abstract, then
answered by experts who marked the paragraphs the answer rests on. The answers
are short: an extracted span, a yes or a no, a sentence, or "unanswerable".
That is this system's shape exactly, and it gives the two things needed to
separate the stages:

```text
  Answer F1    token overlap with the reference answer, best over annotators
  Evidence F1  did the system put the right PARAGRAPHS in front of the model
```

Evidence F1 is the same question as Recall@k, asked in the benchmark's own
terms, so retrieval and answering are scored apart without any extra
machinery.

#### How a paper is indexed
Each paragraph of the full text becomes one unit, carrying its section name,
and goes through the ordinary prose path — the same chunker, embedder and
hybrid retriever the application uses. Because a prose chunk never crosses a
unit boundary, every retrieved chunk maps back to exactly one paragraph, which
is what the evidence metric expects.

Scoring is the benchmark's own evaluator, vendored unchanged. Paragraphs
marked "FLOAT SELECTED" are the captions of figures and tables; they are
dropped from both the gold evidence and ours, which is what the official
--text_evidence_only setting does and the honest setting for a system that
reads no figures.

In [3]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook', '--papers', '0']

In [4]:
import json
import os
import random
import statistics
import sys
import time
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import numpy as np  # noqa: E402

from backend.pipeline import generator, sparse as sparse_module  # noqa: E402
from backend.pipeline.embedder import embed, model_key  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.retriever import DENSE_WEIGHT, retrieve  # noqa: E402
from backend.scripts.vendor.qasper_evaluator import (  # noqa: E402
    evaluate, get_answers_and_evidence)

DATA_DIR = ROOT / "data" / "raw" / "qasper"
EVAL_DIR = ROOT / "data" / "eval"

SEED = 13
CHUNKING = "sentence"
FLOAT_MARKER = "FLOAT SELECTED"
UNANSWERABLE = "Unanswerable"
# What the short style says when it declines, in any of its spellings.
DECLINED = {"unanswerable", "not answerable", "no answer", "n/a", "none", ""}

In [5]:
def arg(flag, default):
    return type(default)(sys.argv[sys.argv.index(flag) + 1]) if flag in sys.argv else default

In [6]:
PAPERS = arg("--papers", 30)          # 0 means the whole dev set
TOP_K = arg("--k", 3)
SPLIT = arg("--split", "dev")
MODEL_TAG = ("" if generator.MODEL_NAME == "Qwen/Qwen2.5-1.5B-Instruct"
             else "_" + generator.MODEL_NAME.split("/")[-1].lower()
             .replace("-instruct", "").replace(":", "-"))
OUT_PATH = EVAL_DIR / ("qasper"
                       + (f"_{PAPERS}papers" if PAPERS else "_dev")
                       + MODEL_TAG
                       + ("" if TOP_K == 3 else f"_k{TOP_K}")
                       + ("" if generator.ABSTAIN == "check"
                          else f"_abstain-{generator.ABSTAIN}")
                       + ".json")

In [7]:
def paragraphs_of(paper):
    """
    Every paragraph of the full text as one indexed unit, with its section.
    Figure and table captions are kept out: this system does not read them,
    and the official text-evidence-only setting excludes them from the gold.
    """
    units = []
    for section in paper["full_text"]:
        name = (section.get("section_name") or "").strip() or None
        for text in section["paragraphs"]:
            body = " ".join(str(text).split())
            if not body or FLOAT_MARKER in body:
                continue
            units.append({"source_file": "paper", "page": len(units) + 1,
                          "section": name, "text": body})
    return units

In [8]:
def build(units):
    chunks = preprocess(units, chunking=CHUNKING)
    if not chunks:
        return None
    vectors = embed(chunks)
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    return chunks, index, sparse_module.build_index(chunks)

In [9]:
def main():
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

    path = DATA_DIR / f"qasper-{SPLIT}-v0.3.json"
    if not path.exists():
        raise SystemExit(f"{path} is missing — extract the QASPER tarballs first")
    data = json.loads(path.read_text(encoding="utf-8"))

    ids = sorted(data)
    if PAPERS:
        rng = random.Random(SEED)
        rng.shuffle(ids)
        ids = ids[:PAPERS]
    chosen = {i: data[i] for i in ids}
    questions = sum(len(p["qas"]) for p in chosen.values())
    print(f"QASPER {SPLIT}: {len(chosen)} of {len(data)} papers, "
          f"{questions} of {sum(len(p['qas']) for p in data.values())} questions")
    print(f"model {generator.MODEL_NAME}   embedder {model_key()}   k={TOP_K}   "
          f"abstain={generator.ABSTAIN}\n")

    predictions, records = {}, []
    started = time.time()
    for n, (paper_id, paper) in enumerate(chosen.items(), start=1):
        units = paragraphs_of(paper)
        built = build(units)
        if built is None:
            print(f"  [{n}/{len(chosen)}] {paper_id}: no text")
            continue
        chunks, index, keywords = built
        paragraph_of = {u["page"]: u["text"] for u in units}

        for qa in paper["qas"]:
            got = retrieve(qa["question"], index, chunks, k=TOP_K,
                           sparse=keywords, dense_weight=DENSE_WEIGHT)
            evidence, seen = [], set()
            for chunk in got:
                text = paragraph_of.get(getattr(chunk, "page", None))
                if text and text not in seen:
                    seen.add(text)
                    evidence.append(text)
            answer = generator.answer_short(qa["question"],
                                            [str(c) for c in got])
            tidy = " ".join(str(answer).split()).strip().strip(".")
            if tidy.lower() in DECLINED:
                tidy, evidence = UNANSWERABLE, []
            predictions[qa["question_id"]] = {"answer": tidy,
                                              "evidence": evidence}
            records.append({"paper_id": paper_id, "question": qa["question"],
                            "predicted": tidy,
                            "evidence_paragraphs": len(evidence)})
        print(f"  [{n}/{len(chosen)}] {paper['title'][:52]:<52} "
              f"{len(units):>4} paras {len(chunks):>4} chunks "
              f"{len(paper['qas']):>2} q  ({time.time() - started:.0f}s)")

    gold = get_answers_and_evidence(chosen, True)   # text evidence only
    scores = evaluate(gold, predictions)

    declined = sum(1 for p in predictions.values() if p["answer"] == UNANSWERABLE)
    gold_unanswerable = sum(
        1 for refs in gold.values()
        if any(r["answer"] == UNANSWERABLE for r in refs))

    print(f"\n  Answer F1    {scores['Answer F1']:.4f}")
    print(f"  Evidence F1  {scores['Evidence F1']:.4f}")
    print("  Answer F1 by type")
    for kind, value in scores["Answer F1 by type"].items():
        print(f"    {kind:<12} {value:.4f}")
    print(f"\n  the system declined {declined} of {len(predictions)} "
          f"({declined / len(predictions):.0%}); "
          f"{gold_unanswerable} are unanswerable in the gold")

    OUT_PATH.write_text(json.dumps({
        "note": "The system on QASPER (Dasigi et al., 2021), scored with the "
                "benchmark's own evaluator, vendored unchanged. Each paragraph "
                "of a paper is one unit through the ordinary prose path, so a "
                "retrieved chunk maps to exactly one paragraph and Evidence F1 "
                "is Recall@k in the benchmark's terms. Figure and table "
                "captions are excluded from both sides, matching the official "
                "text-evidence-only setting.",
        "split": SPLIT,
        "papers": len(chosen),
        "questions": len(predictions),
        "model": generator.MODEL_NAME,
        "backend": generator.BACKEND,
        "embedder": model_key(),
        "chunking": CHUNKING,
        "k": TOP_K,
        "abstain": generator.ABSTAIN,
        "answer_f1": scores["Answer F1"],
        "evidence_f1": scores["Evidence F1"],
        "answer_f1_by_type": scores["Answer F1 by type"],
        "missing_predictions": scores["Missing predictions"],
        "declined": declined,
        "gold_unanswerable": gold_unanswerable,
        "seconds": round(time.time() - started, 1),
        "predictions": predictions,
        "records": records,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

The options above run the whole dev set (`--papers 0`). `SA_BACKEND=ollama` with `SA_MODEL=qwen3:14b` puts the larger model through the same pipeline (notebook 07 compares them).

In [10]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### QASPER results

Every saved run, including the larger model and the abstention settings. `abstain = check` asks the model a separate yes/no question before answering and declines on *no*; `off` always answers.

In [11]:
rows = []
for path in sorted(EVAL_DIR.glob("qasper_*.json")):
    if "baselines" in path.name:
        continue
    r = saved(path.name)
    rows.append({"run": path.stem, "papers": r["papers"], "questions": r["questions"],
                 "model": r["model"], "abstain": r["abstain"],
                 "answer F1": round(r["answer_f1"], 3),
                 "evidence F1": round(r["evidence_f1"], 3),
                 "declined": r["declined"], "gold unanswerable": r["gold_unanswerable"]})
pd.DataFrame(rows).set_index("run")

,papers,questions,model,abstain,answer F1,evidence F1,declined,gold unanswerable
run,,,,,,,,
qasper_30papers,30,99,Qwen/Qwen2.5-1.5B-Instruct,check,0.289,0.307,32,20
qasper_30papers_abstain-off,30,99,Qwen/Qwen2.5-1.5B-Instruct,off,0.265,0.225,0,20
qasper_30papers_qwen3-14b,30,99,qwen3:14b,check,0.400,0.322,30,20
qasper_5papers,5,20,Qwen/Qwen2.5-1.5B-Instruct,check,0.192,0.358,11,1
qasper_dev,281,1005,Qwen/Qwen2.5-1.5B-Instruct,check,0.250,0.292,253,135
qasper_dev_qwen3-14b,281,1005,qwen3:14b,check,0.336,0.311,351,135
qasper_dev_qwen3-14b_abstain-off,281,1005,qwen3:14b,off,0.339,0.248,28,135


In [12]:
dev = saved("qasper_dev.json")
pd.Series(dev["answer_f1_by_type"], name="answer F1 by answer type (Qwen2.5-1.5B, dev)").round(3)

extractive     0.186
abstractive    0.092
boolean        0.550
none           0.643
Name: answer F1 by answer type (Qwen2.5-1.5B, dev), dtype: float64

### Against the published baselines

The paper's numbers (Dasigi et al., 2021), in percent. Their models are fine-tuned on QASPER; ours are zero-shot.

In [13]:
base = saved("qasper_published_baselines.json")
answer = pd.DataFrame(base["answer_f1"]).drop(columns="note", errors="ignore").T
evidence = pd.DataFrame(base["evidence_f1"]).drop(columns="note", errors="ignore").T
ours = pd.DataFrame(base["this_project_dev"]).drop(columns="note", errors="ignore").T
display(answer.rename_axis("published answer F1"))
display(evidence.rename_axis("published evidence F1"))
display(ours.rename_axis("this project, dev"))

,dev,test
published answer F1,,
q_only,17.81,22.48
q_plus_abstract,18.60,22.30
q_plus_introduction,18.30,24.08
q_plus_full_text,29.05,32.80
q_plus_full_text_with_scaffold,28.01,33.63


,dev,test
published evidence F1,,
led_base,23.94,29.85
led_large,31.25,39.37
tf_idf,10.68,9.20
random_paragraph,2.09,1.30
first_paragraph,0.71,0.34
human_lower_bound,NaN,71.62


,answer_f1,evidence_f1
"this project, dev",,
qwen2.5-1.5b,24.99,29.18
qwen3-14b,33.59,31.10
qwen3-14b_abstain_off,33.94,24.76


## Our 25 questions: retrieval or generation failure

Splits wrong answers into retrieval failures and generation failures.

For each ground-truth question the full pipeline runs end to end at the same k
the web app uses (TOP_K = 3), and two independent facts are recorded:

```text
  retrieved_correct : was a chunk matching the expected source_file + page
                      among the chunks actually handed to the generator
  answer_correct    : does the generated text carry the ground-truth answer
```

A third fact separates the two ways bucket 2 can happen:

```text
  answer_in_context : does the ground-truth answer appear verbatim (letters and
                      digits only) in the retrieved text. Only meaningful for
                      extractive answers, so it is None when the answer is not
                      verbatim anywhere on its expected page. A matching page
                      does not guarantee this — the answer can sit in another
                      chunk of the same page.
```

Crossing the first two gives four buckets:

```text
  1. retrieved + correct answer   working as intended
  2. retrieved + wrong answer     GENERATION failure - the evidence was in the
                                  prompt and the model still got it wrong
  3. not retrieved + wrong answer RETRIEVAL failure - the model never saw it
  4. not retrieved + correct answer answered without the supporting chunk
                                  (parametric knowledge, or the fact appears
                                  elsewhere in the corpus)
```

Read-only: retrieval, embedding, chunking and generation are all called exactly
as the app calls them. Nothing is tuned, and no fix is attempted.

In [14]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook', '--chunking', 'sentence', '--retrieval', 'hybrid']

In [15]:
from eval_common import (_FUNCTION_WORDS, grounded_share, judge,  # noqa: F401
                         normalise, numbers_in, squash, token_f1)

In [16]:
import json
import os
import re
import sys
import time
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import numpy as np  # noqa: E402

from backend.pipeline.loader import load_file  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.embedder import embed, model_key  # noqa: E402
from backend.pipeline.retriever import DENSE_WEIGHT, retrieve  # noqa: E402
from backend.pipeline import sparse  # noqa: E402
from backend.pipeline.generator import (ANSWER_STYLE,  # noqa: E402
                                        CHAT_MODEL_NAME, MODEL_NAME, generate)

RAW_DIR = ROOT / "data" / "raw"
EVAL_DIR = ROOT / "data" / "eval"
GROUND_TRUTH_PATH = EVAL_DIR / "retrieval_ground_truth.json"
# "--chunking sentence" measures preprocess(chunking="sentence") instead.
CHUNKING = (sys.argv[sys.argv.index("--chunking") + 1]
            if "--chunking" in sys.argv else "window")
EMBEDDER = model_key()
# "--retrieval hybrid" mixes BM25 keyword scores into retrieval (retriever.py);
# "--retrieval keyword" uses BM25 alone.
RETRIEVAL = (sys.argv[sys.argv.index("--retrieval") + 1]
             if "--retrieval" in sys.argv else "dense")
# SA_ANSWER_STYLE=explain measures the paragraph answers the chat view gives
# instead of the extractive spans (generator.ANSWER_STYLE).
# The answering model is part of the configuration too: results recorded
# before 2026-09-21 were taken on flan-t5-large and carry no model suffix.
# Matches backend/service.py's TOP_K — this measures the configuration that
# actually ships. "--k N" answers the same questions with N chunks instead,
# which is how the choice of 3 is checked rather than assumed.
TOP_K = (int(sys.argv[sys.argv.index("--k") + 1]) if "--k" in sys.argv else 3)
GEN_MODEL_NAME = CHAT_MODEL_NAME if ANSWER_STYLE == "explain" else MODEL_NAME
MODEL_TAG = "" if "flan-t5" in GEN_MODEL_NAME else (
    "_" + GEN_MODEL_NAME.split("/")[-1].lower().replace("-instruct", ""))
OUT_PATH = EVAL_DIR / ("generation_analysis"
                       + ("" if CHUNKING == "window" else f"_{CHUNKING}")
                       + ("" if EMBEDDER == "minilm" else f"_{EMBEDDER}")
                       + ("" if RETRIEVAL == "dense" else f"_{RETRIEVAL}")
                       + ("" if ANSWER_STYLE == "short" else f"_{ANSWER_STYLE}")
                       + MODEL_TAG
                       + ("" if TOP_K == 3 else f"_k{TOP_K}")
                       + ".json")

DOCUMENTS = [
    "embedding.pdf",
    "Whisper.pdf",
    "Flant5pdf.pdf",
    "Hallucinations_in_Large_Language_Models_LLMs.pdf",
]


BUCKETS = {
    (True, True): "1. retrieved + correct answer",
    (True, False): "2. retrieved + wrong answer (generation failure)",
    (False, False): "3. not retrieved + wrong answer (retrieval failure)",
    (False, True): "4. not retrieved + correct answer",
}

In [17]:
def build_index():
    chunks = []
    for name in DOCUMENTS:
        chunks.extend(preprocess(load_file(str(RAW_DIR / name)), chunking=CHUNKING))
    embeddings = embed(chunks)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.ascontiguousarray(embeddings, dtype=np.float32))
    return index, chunks

In [18]:
def main():
    # Passages carry ligatures and dashes that a cp1252 console cannot print,
    # and losing a finished run to a print statement is a poor trade.
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
    print("=" * 78)
    print(f"GENERATION vs RETRIEVAL FAILURE ANALYSIS  (k={TOP_K}"
          + (", as shipped)" if TOP_K == 3 else ")"))
    print("=" * 78)

    ground_truth = json.loads(GROUND_TRUTH_PATH.read_text(encoding="utf-8"))
    print(f"Ground truth: {len(ground_truth)} questions")

    index, chunks = build_index()
    bm25 = sparse.build_index(chunks) if RETRIEVAL != "dense" else None
    print(f"Combined index: {index.ntotal} chunks across {len(DOCUMENTS)} documents\n")

    results = []
    counts = {label: 0 for label in BUCKETS.values()}

    for i, entry in enumerate(ground_truth, start=1):
        expected = (entry["source_file"], entry["page"])

        started = time.time()
        retrieved = retrieve(entry["question"], index, chunks, k=TOP_K, sparse=bm25,
                             dense_weight=0.0 if RETRIEVAL == "keyword" else DENSE_WEIGHT)
        answer = generate(entry["question"], retrieved)
        elapsed = time.time() - started

        retrieved_records = [{
            "rank": r,
            "source_file": c.source_file,
            "page": c.page,
            "is_expected": (c.source_file, c.page) == expected,
            "text": str(c),
        } for r, c in enumerate(retrieved, start=1)]

        retrieved_correct = any(rec["is_expected"] for rec in retrieved_records)
        page_text = squash(" ".join(str(c) for c in chunks
                                    if (c.source_file, c.page) == expected))
        answer_in_context = (squash(entry["answer"]) in squash(" ".join(map(str, retrieved)))
                             if squash(entry["answer"]) in page_text else None)
        answer_correct, signals = judge(entry["answer"], answer)
        bucket = BUCKETS[(retrieved_correct, answer_correct)]
        counts[bucket] += 1

        results.append({
            "index": i,
            "question": entry["question"],
            "expected_answer": entry["answer"],
            "expected_source": {"source_file": expected[0], "page": expected[1]},
            "retrieved_correct": retrieved_correct,
            "answer_in_context": answer_in_context,
            "generated_answer": answer,
            "answer_correct": answer_correct,
            "grounded_share": grounded_share(answer, " ".join(map(str, retrieved))),
            "bucket": bucket,
            "match_signals": signals,
            "seconds": round(elapsed, 2),
            "retrieved_chunks": retrieved_records,
        })

        flag = "R+" if retrieved_correct else "R-"
        flag += "A+" if answer_correct else "A-"
        print(f"  {i:>2}/{len(ground_truth)}  [{flag}] {elapsed:>5.1f}s  "
              f"{expected[0][:26]:<26} p.{expected[1]:<3} "
              f"expected={entry['answer'][:34]!r} got={answer[:34]!r}", flush=True)

    # ---- buckets -------------------------------------------------------------
    print()
    print("=" * 78)
    print("BUCKETS")
    print("=" * 78)
    total = len(results)
    for label in BUCKETS.values():
        n = counts[label]
        print(f"  {n:>2}/{total}  ({n / total:>5.1%})  {label}")

    review = [r for r in results if r["match_signals"]["needs_human_review"]]
    if review:
        print(f"\n  {len(review)} answer(s) sit near the grading threshold and are "
              f"worth eyeballing (flagged in the JSON):")
        for r in review:
            print(f"    Q{r['index']}: expected {r['expected_answer']!r} "
                  f"got {r['generated_answer']!r} (F1 {r['match_signals']['token_f1']})")

    # ---- bucket 2 detail -----------------------------------------------------
    bucket2 = [r for r in results
               if r["bucket"].startswith("2.")]
    print()
    print("=" * 78)
    print(f"BUCKET 2 DETAIL - correct chunk WAS retrieved, answer still wrong "
          f"({len(bucket2)} case(s))")
    print("=" * 78)
    if not bucket2:
        print("  none")
    for r in bucket2:
        print()
        print("-" * 78)
        print(f"Q{r['index']}. {r['question']}")
        print(f"  expected answer : {r['expected_answer']!r}")
        print(f"  generated answer: {r['generated_answer']!r}")
        print(f"  token F1        : {r['match_signals']['token_f1']}")
        print(f"  chunks passed to the generator:")
        for rec in r["retrieved_chunks"]:
            mark = ">> EXPECTED" if rec["is_expected"] else "           "
            print(f"    {mark} #{rec['rank']} {rec['source_file']} p.{rec['page']}")
            body = " ".join(rec["text"].split())
            print(f"        {body}")

    payload = {
        "note": "Read-only analysis. Retrieval, embedding, chunking and "
                "generation are unchanged; no fix attempted.",
        "k": TOP_K,
        "chunking": CHUNKING,
        "embedder": EMBEDDER,
        "answer_style": ANSWER_STYLE,
        "model": GEN_MODEL_NAME,
        "retrieval": RETRIEVAL,
        "total_questions": total,
        "bucket_counts": counts,
        "bucket_2_answer_in_context": {
            str(k): sum(1 for r in results if r["bucket"].startswith("2.")
                        and r["answer_in_context"] is k)
            for k in (True, False, None)},
        "grading_rule": "An answer counts as correct if the normalised expected "
                        "answer is contained in the output (or vice versa), or "
                        "token F1 >= 0.6, or every number in the expected answer "
                        "appears in the output. All signals are stored per "
                        "question so calls can be re-judged.",
        "results": results,
    }
    OUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
                        encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

The options above are the shipped configuration: sentence chunking, bge-small, hybrid retrieval, k = 3, Qwen2.5-1.5B.

In [19]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [20]:
ga = saved("generation_analysis_sentence_bge-small_hybrid_qwen2.5-1.5b.json")
print(f"{ga['total_questions']} questions, k={ga['k']}, {ga['model']}, "
      f"{ga['embedder']}, {ga['retrieval']} retrieval")
display(pd.Series(ga["bucket_counts"], name="questions").to_frame())
table = pd.DataFrame([{"question": r["question"], "expected": r["expected_answer"],
                       "answer": r["generated_answer"], "retrieved": r["retrieved_correct"],
                       "correct": r["answer_correct"]} for r in ga["results"]])
table[~table["correct"]]

25 questions, k=3, Qwen/Qwen2.5-1.5B-Instruct, bge-small, hybrid retrieval


,questions
1. retrieved + correct answer,16
2. retrieved + wrong answer (generation failure),5
3. not retrieved + wrong answer (retrieval failure),2
4. not retrieved + correct answer,2


,question,expected,answer,retrieved,correct
0,What is the size of the synthetic dataset used...,"3,197 sentence pairs",8%,False,False
5,"In Figure 3 of the FLAN paper, what four combi...","with/without exemplars, crossed with with/with...","unk, t0-sf, niv2, cot",True,False
10,What learning rate was used for the MNRL-only ...,2e-5,1e-5,True,False
14,How many hours of speech translation data does...,"125,000 hours of X-to-English translation data","117,000",True,False
15,How many parameters does the Whisper Large mod...,1550M parameters,540 billion,True,False
20,How many tokens are in PaLM's pre-training dat...,780B pre-training tokens vs 1.4B finetuning to...,540b,False,False
23,How does the paper define overfitting as a cau...,the model learns the training data too well an...,increased complexity in models with more layer...,True,False
